In [0]:
from pyspark.sql import functions as F
def process_books_sales():
    orders_df = spark.readStream.table("dev.silver.orders_silver").withColumn("book", F.explode(F.col("books"))) 

    books_df = spark.read.table("dev.silver.books_silver")

    sql_query = (
        orders_df.join(
            books_df,
            orders_df.book.book_id == books_df.book_id,
            "inner"
        ).writeStream
            .outputMode("append")
            .option("checkpointLocation", "dbfs:/Volumes/dev/pro_landing_zone/checkpoints/books_sales")
            .trigger(availableNow=True)
            .table("dev.silver.books_sales")
    )
    sql_query.awaitTermination()
process_books_sales()


In [0]:
%sql
select * from dev.silver.books_sales

In [0]:
def process_current_books():
    spark.sql("""
        CREATE OR REPLACE TABLE dev.silver.current_books
        AS 
        SELECT 
            book_id, 
            title,
            author, 
            price
        FROM dev.silver.books_sales
        WHERE current IS TRUE
    """)
process_current_books()

In [0]:
%sql
select * from dev.silver.current_books